# Finance Quiz Solutions Notebook

This notebook calculates and explains the following questions:

1. **Term structure of interest rates and swap valuation**  
2. **Swap rates**  
3. **Hedging using futures**  
4. **Call options**  
5. **Call options II (replicating portfolio cash account)**

All formulas are written in markdown, and each question also includes Python code to verify the result.


In [ ]:
import math
import pandas as pd

## Question 1 — Term structure of interest rates

Suppose the annual-compounded spot rates are:

- $s_1 = 7.0\%$
- $s_2 = 7.3\%$
- $s_3 = 7.7\%$
- $s_4 = 8.1\%$
- $s_5 = 8.4\%$
- $s_6 = 8.8\%$

We want to compute the discount factor $d(0,4)$.

With annual compounding, the discount factor is

$$
d(0,t) = \frac{1}{(1+s_t)^t}
$$

So for $t=4$,

$$
d(0,4) = \frac{1}{(1+0.081)^4}
$$


In [ ]:
spot_rates = {
    1: 0.070,
    2: 0.073,
    3: 0.077,
    4: 0.081,
    5: 0.084,
    6: 0.088,
}

discount_factors = {t: 1 / ((1 + s) ** t) for t, s in spot_rates.items()}
discount_factors[4]

### Answer

$$
d(0,4) = 0.7323
$$

Rounded to three decimal places:

$$
\boxed{0.732}
$$


## Question 2 — Swap rate

We now use the term structure from Question 1 to find the fixed rate on a **6-year plain vanilla swap** that makes the swap value equal to zero.

For annual payments, the fair fixed swap rate is

$$
r_{swap} = \frac{1 - d(0,6)}{\sum_{t=1}^{6} d(0,t)}
$$

The notional principal of $10$ million does **not** affect the fixed rate itself. It would only scale the dollar cash flows.


In [ ]:
df_discount = pd.DataFrame(
    {
        "t": list(discount_factors.keys()),
        "spot_rate": [spot_rates[t] for t in discount_factors.keys()],
        "discount_factor": [discount_factors[t] for t in discount_factors.keys()],
    }
)
df_discount

In [ ]:
d06 = discount_factors[6]
swap_rate = (1 - d06) / sum(discount_factors.values())
swap_rate

### Answer

Using

$$
r_{swap} = \frac{1 - d(0,6)}{\sum_{t=1}^{6} d(0,t)}
$$

we get

$$
r_{swap} = 0.0862 = 8.62\%
$$

So the answer is

$$
\boxed{8.62\%}
$$


## Question 3 — Hedging using futures

A farmer expects to sell **150,000 pounds** of orange juice in 3 months.

Each futures contract covers **15,000 pounds** and the current futures price is

$$
F_0 = 118.65
$$

in cents per pound.

### Step 1: Number of contracts

$$
N = \frac{150{,}000}{15{,}000} = 10
$$

So the farmer should **short 10 futures contracts**.

### Step 2: Locked-in price

Because the hedge quantity matches exactly and the farmer can meet all margin calls, the futures hedge removes price uncertainty and locks in the current futures price.

So the guaranteed price is simply

$$
\boxed{118.65}
$$

cents per pound.


In [ ]:
harvest_qty = 150_000
contract_size = 15_000
F0 = 118.65  # cents per pound

num_contracts = harvest_qty / contract_size
locked_price = F0

num_contracts, locked_price

### Answer

Number of contracts:

$$
\boxed{10}
$$

Guaranteed risk-free sale price:

$$
\boxed{118.65}
$$

cents per pound.


## Question 4 — One-period European call option

We are given a 1-period binomial model with:

- $R = 1.02$
- $S_0 = 100$
- $u = 1.05$
- $d = 1/1.05$
- strike $K = 102$

There are no dividends.

### Step 1: Stock values next period

Up-state stock price:

$$
S_u = uS_0 = 1.05 \times 100 = 105
$$

Down-state stock price:

$$
S_d = dS_0 = \frac{1}{1.05} \times 100 = 95.2381
$$

### Step 2: Call payoffs

Up-state payoff:

$$
C_u = \max(105 - 102, 0) = 3
$$

Down-state payoff:

$$
C_d = \max(95.2381 - 102, 0) = 0
$$

### Step 3: Risk-neutral probability

$$
q = \frac{R-d}{u-d}
$$

### Step 4: Option price

$$
C_0 = \frac{1}{R}\left[qC_u + (1-q)C_d\right]
$$


In [ ]:
R = 1.02
S0 = 100
u = 1.05
d = 1 / 1.05
K = 102

Su = u * S0
Sd = d * S0
Cu = max(Su - K, 0)
Cd = max(Sd - K, 0)
q = (R - d) / (u - d)
C0 = (q * Cu + (1 - q) * Cd) / R

pd.DataFrame(
    {
        "quantity": ["Su", "Sd", "Cu", "Cd", "q", "C0"],
        "value": [Su, Sd, Cu, Cd, q, C0],
    }
)

### Answer

The arbitrage-free call value is

$$
C_0 = 2.0373
$$

Rounded to two decimal places:

$$
\boxed{2.04}
$$


## Question 5 — Replicating portfolio cash account

This question refers to the option in Question 4.

We construct a replicating portfolio with:

- $x$ shares of stock
- $y$ dollars in the cash account

The portfolio must match the call payoff in both states:

$$
uS_0x + Ry = C_u
$$

$$
dS_0x + Ry = C_d
$$

Subtract the second equation from the first:

$$
(u-d)S_0x = C_u - C_d
$$

So

$$
x = \frac{C_u - C_d}{(u-d)S_0}
$$

Then plug back into either equation to solve for $y$:

$$
y = \frac{C_u - uS_0x}{R}
$$

or equivalently

$$
y = \frac{C_d - dS_0x}{R}
$$


In [ ]:
x = (Cu - Cd) / ((u - d) * S0)
y = (Cu - u * S0 * x) / R

pd.DataFrame(
    {
        "quantity": ["x (shares of stock)", "y (cash account dollars)"],
        "value": [x, y],
    }
)

### Answer

The replicating portfolio is:

$$
x = 0.3073
$$

and

$$
y = -28.6941
$$

Rounded to three decimal places:

$$
\boxed{-28.694}
$$

The negative sign means you **borrow** \$28.694 from the cash account.


## Final answers summary

1. Discount factor $d(0,4)$:

$$
\boxed{0.732}
$$

2. 6-year swap fixed rate:

$$
\boxed{8.62\%}
$$

3. Hedging using futures:
- Number of contracts: $\boxed{10}$
- Guaranteed price: $\boxed{118.65}$ cents per pound

4. One-period European call option value:

$$
\boxed{2.04}
$$

5. Cash account in the replicating portfolio:

$$
\boxed{-28.694}
$$
